# Audio fingerprinting workbook

## I. Generating Peak Pair Hashes for query and reference files

### 1.1 Constellations

Transform audio file into time-frequency domain

In [ ]:
import sys

sys.path.append('..')

import matplotlib.pyplot as plt
from IPython.display import Audio, HTML
from main import *
from utils import *

Spectrogram parameters

In [ ]:
reference_song_id = '086795'
query_song_id = reference_song_id

sample_rate = SR #44100
window_length = WINDOW_LENGTH #2048
hop_length = HOP_LENGTH #1024
n_fft = N_FFT #2048

reference_spectrogram = to_spectrogram('../audio_data/database/' + reference_song_id + '.mp3', sample_rate, window_length, hop_length, n_fft)
plt.pcolormesh(reference_spectrogram)

### 1.2 Find features – prominent peaks

Peak filter parameters

In [ ]:
peak_level_threshold_db = 2
neighbourhood_size = 20

In [ ]:
# how we define prominent peaks (in constellation())
# local_max = maximum_filter(spectrogram, mode="constant", size=neighbourhood_size)
# is_peak = (spectrogram == local_max) & (spectrogram > level_threshold)

In [ ]:
reference_peaks = constellation(reference_spectrogram, peak_level_threshold_db, neighbourhood_size)
plt.pcolormesh(reference_spectrogram)
plt.scatter(reference_peaks[:,1], reference_peaks[:,0], marker='+', color='r')


In [ ]:
# # Alternative way to extract peaks
# from skimage.feature import peak_local_max
# min_distance = 10
# peaks = peak_local_max(reference_spectrogram, min_distance=min_distance, threshold_abs=peak_level_threshold_db, exclude_border=False)
# frequency, time = peaks[:, 1], peaks[:, 0]
# plt.pcolormesh(reference_spectrogram)
# plt.scatter(frequency, time, marker='x', color='r')


### 1.3 Generate peak pairing hashes

pairing hash filter constants

In [ ]:
fan_out = 10
min_time_delta = 1
max_time_delta = 200
max_frequency_delta = 2000
max_freq_delta = round(max_frequency_delta * n_fft / sample_rate) # 93

In [ ]:
reference_hashes = generate_hashes(reference_peaks, fan_out, min_time_delta, max_time_delta, max_freq_delta)

# hash: [hash_val, t_anchor]
# hash_val = (int(f_anchor & 0x3FF) << 20) | (int(f_target & 0x3FF) << 10) | int(delta_t & 0x3FF)

hash_index = 1

reference_hash = reference_hashes[hash_index]
t_anchor = reference_hash[1]

# decode information stored in hash
f_anchor = reference_hash[0] >> 20
f_target = reference_hash[0] >> 10 & 0x3FF
delta_t = reference_hash[0] & 0xFF
t_target = t_anchor + delta_t


plt.scatter(reference_peaks[:,1], reference_peaks[:,0])
plt.scatter([t_anchor, t_target], [f_anchor, f_target], marker='x')


## 2. Match query Files

let's have a look at the constellations

In [ ]:
spectrogram = to_spectrogram('../audio_data/query/' + reference_song_id + '_query.mp3', sample_rate, window_length, hop_length, n_fft)

query_peaks = constellation(spectrogram, peak_level_threshold_db, neighbourhood_size)

plt.pcolormesh(spectrogram)
plt.scatter(query_peaks[:,1], query_peaks[:,0], marker='+', color='r')

Here's what it sounds like compared to it's reference file

In [ ]:
query_song = '../audio_data/query/' + reference_song_id + '_query.mp3'
reference_song = '../audio_data/database/' + reference_song_id + '.mp3'

Audio(query_song, rate=sample_rate)


In [ ]:
Audio(reference_song, rate=sample_rate)

We can't match the reference peaks with the query peaks just yet

In [ ]:
plt.scatter(query_peaks[:,1], query_peaks[:,0], marker='+', color='r')
plt.scatter(reference_peaks[:,1], reference_peaks[:,0])

### 2.1 Finding identical hash values in query and reference data

In [ ]:
query_hashes = generate_hashes(query_peaks, fan_out, min_time_delta, max_time_delta, max_freq_delta)
query_hash_values = [hash[0] for hash in query_hashes]
query_min = np.min(query_hash_values)
query_max = np.max(query_hash_values)

reference_hash_values = [hash[0] for hash in reference_hashes if hash[0] > query_min and hash[0] < query_max]

plt.scatter(reference_hash_values, np.zeros_like(reference_hash_values), marker='_')
plt.scatter(query_hash_values, np.zeros_like(query_hash_values), marker='|')


identical_hashes = np.sort([query_hash for query_hash in query_hash_values if query_hash in reference_hash_values])
plt.scatter(identical_hashes, np.zeros_like(identical_hashes), marker='*')


for now, don't consider absolute time to find matches

## 3. Scoring with time delta

absolute time of found peaks is considered in a second step of the algorithm

In [ ]:
ref_fingerprint = {}
fingerprints(reference_hashes, reference_song_id, ref_fingerprint)
matches = np.array(find_matches(query_hashes, ref_fingerprint))
plt.scatter(matches[:,2], matches[:,1])


In [ ]:
time_diff_counts = time_diff_histogram(find_matches(query_hashes, ref_fingerprint))
x_values = [int(key[1]) for key in time_diff_counts]
counts = list(time_diff_counts.values())
max_count_time_diff = max(time_diff_counts, key=time_diff_counts.get)[1]
plt.bar(x_values, counts)
print(max_count_time_diff)

In [ ]:
query_peaks = constellation(spectrogram, peak_level_threshold_db, neighbourhood_size)
plt.scatter(reference_peaks[:,1], reference_peaks[:,0])
plt.scatter(query_peaks[:,1] + max_count_time_diff, query_peaks[:,0], marker='x')

## 4. False Positives

Some results return false positives due to factors degrading audio quality, such as high noise ratios, reverb, and poor recording quality

Ex. <br>
Query: 004850_classical_query.mp3 <br>
Ranked 1st: 57938.mp3 (score: 3) <br>
Actual rank: 3rd place 004850_classical.mp3  (score: 2, starts at: 10.52s) 


In [ ]:
sample_query = "004850_classical_query.mp3"
fp_reference = "057938.mp3"

In [ ]:
sample_rate = 44100
window_length = 4096
hop_length = 2048
n_fft = 4096

peak_level_threshold_db = 2
neighbourhood_size = 20

sample_spectrogram = to_spectrogram('../audio_data/query/' + sample_query, sr=sample_rate, hop_length=hop_length, window_length=window_length, n_fft=n_fft)
sample_peaks = constellation(sample_spectrogram, peak_level_threshold_db, neighbourhood_size)

freq_bin, time_bin = sample_peaks[:, 0], sample_peaks[:, 1]

plt.pcolormesh(sample_spectrogram, cmap='magma')
plt.scatter(x=time_bin,y=freq_bin, marker='x', color='w')



Noise artefacts, such as speech or white noise, are mistaken as correct peaks since noise dilutes energy peaks.

In [ ]:
Audio("../audio_data/query/" + sample_query)


In [ ]:
query = get_peaks_and_hashes('../audio_data/query/' + sample_query)
reference = get_peaks_and_hashes('../audio_data/database/' + fp_reference)
plot_match(query, reference, SR, HOP_LENGTH, N_FFT)
